In [ ]:
# =============================================================================
# CELL 0: SETUP - Install and Import
# =============================================================================
print("Installing required packages...")
!pip install -q transformers sentencepiece accelerate datasets torch

import re
import json
import torch
import textwrap
import traceback
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")


Installing required packages...
✅ Device: cuda


In [ ]:
# Initialize storage
summaries_dict = {"summaries": {}}

def add_summary(k, m, s, status="success", error=None):
    summaries_dict["summaries"][k] = {
        "model": m, "summary": s if status == "success" else None,
        "status": status, "error": str(error) if error else None,
        "summary_length": len(s) if s else 0,
        "timestamp": datetime.now().isoformat()
    }

# Load data

INPUT_FILE = "gcn_sentences.json"

try:
    def get_ordered_summary_text():
        with open(INPUT_FILE, 'r') as f:
            data = json.load(f)

        sentence_list = data.get('summary', [])

        joined_text = "\n\n".join(sentence_list)

        print(f"✅ Loaded {len(sentence_list)} sentences, {len(joined_text)} chars")

        return joined_text

    joined_text = get_ordered_summary_text()


except Exception as e:
    print(f"⚠️ Error loading or parsing JSON: {e}")
    joined_text = "The court issued a consent decree to resolve all claims."

print("✅ Setup complete")


# =============================================================================
# CELL 1: HELPER FUNCTIONS
# =============================================================================

def clean_legal_boilerplate(text):
    """Remove legal document artifacts"""
    # Case headers
    text = re.sub(r'Case\s+\d+[:\d\w\-\.]+\s+Document\s+\d+.*?Filed.*?\d{4}', '', text, flags=re.I)
    text = re.sub(r'IN THE UNITED STATES DISTRICT.*?DIVISION', '', text, flags=re.I|re.DOTALL)
    text = re.sub(r'Civil Action No\..*?\n', '', text, flags=re.I)

    # Attorney info
    text = re.sub(r'(ATTORNEY|LEAD ATTORNEY|Senior Trial Attorney).*?(?=\n\n|\Z)', '', text, flags=re.I|re.DOTALL)
    text = re.sub(r'(Phone|Telephone|Facsimile|Email|Fax)\s*:?\s*[\(\d\)\s\-\.]+', '', text, flags=re.I)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', '', text)
    text = re.sub(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', '', text)
    text = re.sub(r'Bar\s+ID\s+\d+', '', text, flags=re.I)

    # Addresses
    text = re.sub(r'\d+\s+\w+\s+(Street|St\.|Avenue|Ave\.|Road|Rd\.|Suite|Ste\.).*?\d{5}', '', text, flags=re.I)
    text = re.sub(r'\b[A-Z][a-z]+,\s+[A-Z]{2}\s+\d{5}\b', '', text)

    # Signatures
    text = re.sub(r'Respectfully submitted.*?(?=\n\n|\Z)', '', text, flags=re.I|re.DOTALL)
    text = re.sub(r'/s/.*?\n', '', text)

    # Page numbers
    text = re.sub(r'\bPage\s+\d+\s+of\s+\d+\b', '', text, flags=re.I)
    text = re.sub(r'Exhibit\s+[A-Z\d]+', '', text, flags=re.I)

    # Random numbers
    text = re.sub(r'\b\d{4,}\b(?!\s+dollars)', '', text)
    text = re.sub(r'(?<!\w)-\d+-(?!\w)', '', text)

    # Fix spacing
    text = re.sub(r'(\w)-\s+(\w)', r'\1\2', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Boilerplate phrases
    text = re.sub(r'incorporates?\s+by\s+reference.*?(?=\.|$)', '', text, flags=re.I)

    return text.strip()

def extract_substantive_content(text):
    """Keep only meaningful legal content"""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    substantive = []

    substantive_keywords = [
        r'\b(plaintiff|defendant|court|alleged|claims?|violation|discriminat|breach|contract|employment|terminat|injunction)\b',
        r'\b(relief|damages|compensa|settlement|decree|judgment|ruling|finding|holding)\b',
        r'\b(unlawful|illegal|violated|engaged in|failed to|refused to)\b',
        r'\b(argues?|contends?|maintains?|asserts?|alleges?)\b'
    ]

    exclude_patterns = [
        r'^\d+\.',
        r'^[A-Z]\.',
        r'\b(filed|docketed|served|received)\b.*\d{4}',
        r'pursuant to',
    ]

    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 20:
            continue

        is_substantive = any(re.search(p, sent, re.I) for p in substantive_keywords)
        is_excluded = any(re.search(p, sent, re.I) for p in exclude_patterns)

        if is_substantive and not is_excluded:
            substantive.append(sent)

    return ' '.join(substantive)

def smart_chunk_text(text, tokenizer, max_tokens=800, overlap_tokens=100):
    """Chunk with overlap"""
    tokens = tokenizer.encode(text, add_special_tokens=False)

    if len(tokens) <= max_tokens:
        return [text]

    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)

        if end >= len(tokens):
            break
        start = end - overlap_tokens

    return chunks

def post_process_summary(summary):
    """Clean generated summary, removing artifacts, boilerplate, and abrupt endings."""
    # 1. General artifact removal
    summary = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', '', summary)
    summary = re.sub(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', '', summary)
    summary = re.sub(r'Bar\s+ID\s+\d+', '', summary, flags=re.I)
    summary = re.sub(r'\(\d+\)\s*\d+-\d+', '', summary)

    # 2. Boilerplate/Redundancy Removal (Specific to legal text issues seen)
    summary = re.sub(r'A clear and strong message.*?d\. An\.', '', summary, flags=re.DOTALL)
    summary = re.sub(r'A clear and strong message.*?d\. An', '', summary, flags=re.DOTALL)
    summary = re.sub(r'o\. A clear and strong message.*?d\. An', '', summary, flags=re.DOTALL)

    # 3. Aggressive removal of incomplete final sentence/fragment (e.g., "o. A clear and strong message..." or "She.")
    if summary:
        # Check for non-sentence endings (e.g., ending with a partial word fragment or a single letter)
        match = re.search(r'\s+([a-zA-Z]{1,3}\.)?$', summary)
        if match and len(summary.split()) > 10:
            # Remove everything from the last full stop onwards if it looks like a fragment
            last_full_stop = summary.rfind('.')
            if last_full_stop != -1 and len(summary) - last_full_stop < 50:
                 summary = summary[:last_full_stop + 1]

    # 4. Fix spacing
    summary = re.sub(r'\s+([.,;:!?])', r'\1', summary)
    summary = re.sub(r'\s{2,}', ' ', summary)

    # 5. Remove repetitions (Relaxed)
    words = summary.split()
    if len(words) > 10:
        for i in range(len(words) - 5):
            sequence = ' '.join(words[i:i+3])
            rest = ' '.join(words[i+3:])
            if sequence in rest and summary.endswith(sequence):
                summary = ' '.join(words[:i+3])
                break

    # 6. Capitalization and final punctuation
    if summary and summary[0].islower():
        summary = summary[0].upper() + summary[1:]

    if summary and summary[-1] not in '.!?':
        summary += '.'

    return summary.strip()

print("✅ Helper functions loaded")

# =============================================================================
# CELL 2: Prepare Cleaned Text
# =============================================================================

cleaned_text = clean_legal_boilerplate(joined_text)
substantive_text = cleaned_text

print(f"Original: {len(joined_text)} chars")
print(f"Cleaned: {len(cleaned_text)} chars")
print(f"\nFinal Text: \n{substantive_text[:300]} ...")

# Simple chunking
def chunk_text(text, tokenizer, max_len=500, overlap=50):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_len:
        return [text]
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_len, len(tokens))
        chunks.append(tokenizer.decode(tokens[start:end]))
        start = end - overlap
        if start < 0:
            start = 0
        if end >= len(tokens):
            break
    return chunks

#print(substantive_text)

✅ Loaded 50 sentences, 10444 chars
✅ Setup complete
✅ Helper functions loaded
Original: 10444 chars
Cleaned: 10282 chars

Final Text: 
Respondents REPORT The matter above mentioned was listed before the Hon'ble Court on 9/06/ when the Court was pleased to pass the following Order:-Delay condoned. Having possessed the eligibility criteria of being an intermediate (10 2 pass), he also cleared the written examination and the Physical  ...


In [ ]:


# =============================================================================
# CELL 3: MODEL 1 - T5-BASE
# =============================================================================
print("\n" + "="*80)
print("MODEL 1: T5-BASE")
print("="*80)

model_name = "t5-base"
key = "t5-base"
prefix = "summarize: Provide a detailed legal summary. "

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Simple chunking
def chunk_text(text, tokenizer, max_len=500, overlap=50):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_len:
        return [text]
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_len, len(tokens))
        chunks.append(tokenizer.decode(tokens[start:end]))
        start = end - overlap
        if start < 0:
            start = 0
        if end >= len(tokens):
            break
    return chunks

chunks = chunk_text(substantive_text, tokenizer)

summary_chunks = []
for i, chunk in enumerate(chunks):
    inputs = tokenizer(prefix + chunk, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(
        **inputs,
        num_beams=6,
        no_repeat_ngram_size=3,
        repetition_penalty=2.0,
        length_penalty=2.0,
        max_new_tokens=200,
        early_stopping=True
    )
    summary_chunks.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

final_summary = " ".join(summary_chunks)
add_summary(key, model_name, final_summary)
print("\n📝 T5-BASE SUMMARY:\n")
print(textwrap.fill(final_summary, width=100))


# =============================================================================
# CELL 4: MODEL 2 - BART-Large-CNN
# =============================================================================
print("\n" + "="*80)
print("MODEL 2: BART-LARGE-CNN")
print("="*80)

model_name = "facebook/bart-large-cnn"
key = "bart_large_cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

chunks = chunk_text(substantive_text, tokenizer, max_len=600, overlap=50)
summary_chunks = []

for chunk in chunks:
    inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=1024).to(device)
    outputs = model.generate(
        **inputs,
        num_beams=6,
        no_repeat_ngram_size=3,
        repetition_penalty=2.0,
        length_penalty=2.0,
        max_new_tokens=200,
        early_stopping=True
    )
    summary_chunks.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

final_summary = " ".join(summary_chunks)
add_summary(key, model_name, final_summary)
print("\n📝 BART-LARGE-CNN SUMMARY:\n")
print(textwrap.fill(final_summary, width=100))


# =============================================================================
# CELL 5: MODEL 3 - Pegasus-XSum
# =============================================================================
print("\n" + "="*80)
print("MODEL 3: PEGASUS-XSUM")
print("="*80)

model_name = "google/pegasus-xsum"
key = "pegasus_xsum"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

chunks = chunk_text(substantive_text, tokenizer, max_len=450, overlap=50)
summary_chunks = []

for chunk in chunks:
    inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(
        **inputs,
        num_beams=6,
        no_repeat_ngram_size=3,
        repetition_penalty=2.0,
        length_penalty=2.0,
        max_new_tokens=200,
        early_stopping=True
    )
    summary_chunks.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

final_summary = " ".join(summary_chunks)
add_summary(key, model_name, final_summary)
print("\n📝 PEGASUS-XSUM SUMMARY:\n")
print(textwrap.fill(final_summary, width=100))

# =============================================================================
# CELL 6: MODEL 4 - LED-Base-16384 (LONG CONTEXT)
# =============================================================================
print("\n" + "="*80)
print("MODEL 4: LED-BASE-16384")
print("="*80)

model_name = "allenai/led-base-16384"
key = "led_base_16384"
prefix = "" # No prefix for LED to avoid boilerplate output

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    # LED can handle longer inputs
    chunks = smart_chunk_text(substantive_text, tokenizer, max_tokens=2000, overlap_tokens=200)
    print(f"Created {len(chunks)} chunks")

    gen_args = {
        "num_beams": 8, # Increased beams
        "no_repeat_ngram_size": 3,
        "repetition_penalty": 2.5,
        "length_penalty": 2.5,
        "early_stopping": True,
        "max_new_tokens": 600, # Increased token limit
    }

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}...")
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True,
                          max_length=4096, padding=False).to(device)

        outputs = model.generate(**inputs, **gen_args)
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summary = post_process_summary(summary)
        chunk_summaries.append(summary)

    if len(chunk_summaries) > 1:
        combined = " ".join(chunk_summaries)
        inputs = tokenizer(combined, return_tensors="pt", truncation=True,
                          max_length=2048).to(device)
        outputs = model.generate(**inputs, **gen_args)
        final_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:
        final_summary = chunk_summaries[0]

    final_summary = post_process_summary(final_summary)
    add_summary(key, model_name, final_summary)

    print(f"\n📝 LED Summary ({len(final_summary)} chars):")
    print(textwrap.fill(final_summary, width=100))

    del model, tokenizer
    torch.cuda.empty_cache()

except Exception as e:
    print(f"❌ Error: {e}")
    traceback.print_exc()
    add_summary(key, model_name, None, status="failed", error=e)

# =============================================================================
# CELL 7: MODEL 5 - Legal-Pegasus (BEST PERFORMER)
# =============================================================================
print("\n" + "="*80)
print("MODEL 5: LEGAL-PEGASUS")
print("="*80)

model_name = "nsi319/legal-pegasus"
key = "legal_pegasus"
prefix = "Generate an extremely detailed summary of the legal case. "

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    chunks = smart_chunk_text(substantive_text, tokenizer, max_tokens=450, overlap_tokens=50)
    print(f"Created {len(chunks)} chunks")

    gen_args = {
        "num_beams": 8, # Increased beams
        "no_repeat_ngram_size": 3,
        "repetition_penalty": 2.5,
        "length_penalty": 2.0,
        "early_stopping": True,
        "max_new_tokens": 600, # Increased token limit
    }

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}...")
        prompt = prefix + chunk
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                          max_length=512, padding=False).to(device)

        outputs = model.generate(**inputs, **gen_args)
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summary = post_process_summary(summary)
        chunk_summaries.append(summary)

    if len(chunk_summaries) > 1:
        combined = " ".join(chunk_summaries)
        meta_prompt = prefix + combined
        inputs = tokenizer(combined, return_tensors="pt", truncation=True,
                          max_length=512).to(device)
        outputs = model.generate(**inputs, **gen_args)
        final_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:
        final_summary = chunk_summaries[0]

    final_summary = post_process_summary(final_summary)
    add_summary(key, model_name, final_summary)

    print(f"\n📝 Legal-Pegasus Summary ({len(final_summary)} chars):")
    print(textwrap.fill(final_summary, width=100))

    del model, tokenizer
    torch.cuda.empty_cache()

except Exception as e:
    print(f"❌ Error: {e}")
    traceback.print_exc()
    add_summary(key, model_name, None, status="failed", error=e)

# =============================================================================
# CELL 8: MODEL 6 - Long-T5-TGlobal-Base
# =============================================================================
print("\n" + "="*80)
print("MODEL 6: LONG-T5-TGLOBAL-BASE")
print("="*80)

model_name = "google/long-t5-tglobal-base"
key = "long_t5_tglobal_base"
prefix = "summarize: Provide a detailed and comprehensive legal summary. "

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    chunks = smart_chunk_text(substantive_text, tokenizer, max_tokens=1500, overlap_tokens=150)
    print(f"Created {len(chunks)} chunks")

    gen_args = {
        "num_beams": 8,
        "no_repeat_ngram_size": 3,
        "repetition_penalty": 3.5,
        "length_penalty": 2.5,
        "early_stopping": True,
        "max_new_tokens": 600,
    }

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}...")
        prompt = prefix + chunk
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                          max_length=2048, padding=False).to(device)

        outputs = model.generate(**inputs, **gen_args)
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summary = post_process_summary(summary)
        chunk_summaries.append(summary)

    if len(chunk_summaries) > 1:
        combined = " ".join(chunk_summaries)
        meta_prompt = prefix + combined
        inputs = tokenizer(meta_prompt, return_tensors="pt", truncation=True,
                          max_length=1024).to(device)
        outputs = model.generate(**inputs, **gen_args)
        final_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:
        final_summary = chunk_summaries[0]

    final_summary = post_process_summary(final_summary)
    add_summary(key, model_name, final_summary)

    print(f"\n📝 Long-T5 Summary ({len(final_summary)} chars):")
    print(textwrap.fill(final_summary, width=100))

    del model, tokenizer
    torch.cuda.empty_cache()

except Exception as e:
    print(f"❌ Error: {e}")
    traceback.print_exc()
    add_summary(key, model_name, None, status="failed", error=e)

# =============================================================================
# CELL 9: MODEL 7 - FLAN-T5-Base
# =============================================================================
print("\n" + "="*80)
print("MODEL 7: FLAN-T5-BASE")
print("="*80)

model_name = "google/flan-t5-base"
key = "flan_t5_base"
prefix = "Summarize this legal text in detail: "

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

chunks = chunk_text(substantive_text, tokenizer, max_len=500, overlap=50)
summary_chunks = []

for chunk in chunks:
    inputs = tokenizer(prefix + chunk, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(
        **inputs,
        num_beams=6,
        no_repeat_ngram_size=3,
        repetition_penalty=2.0,
        length_penalty=2.0,
        max_new_tokens=200,
        early_stopping=True
    )
    summary_chunks.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

final_summary = " ".join(summary_chunks)
add_summary(key, model_name, final_summary)
print("\n📝 FLAN-T5-BASE SUMMARY:\n")
print(textwrap.fill(final_summary, width=100))



# =============================================================================
# CELL 10: MODEL - Legal-T5 (BillSum CA/Abstractive)
# =============================================================================
print("\n" + "="*80)
print("MODEL 7: LEGAL-T5 (BILLSUM CA/ABSTRACTIVE)")
print("="*80)

model_name = "stevhliu/t5-small-finetuned-billsum-ca_test"
key = "legal_t5_billsum"
prefix = "summarize: Provide a detailed legal summary. "

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    # Use the existing simple chunker for T5-based models
    chunks = chunk_text(substantive_text, tokenizer, max_len=500, overlap=50)
    print(f"Created {len(chunks)} chunks")

    gen_args = {
        "num_beams": 6,
        "no_repeat_ngram_size": 3,
        "repetition_penalty": 2.0,
        "length_penalty": 2.0,
        "early_stopping": True,
        "max_new_tokens": 200,
    }

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}...")
        prompt = prefix + chunk
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                          max_length=512, padding=False).to(device)

        outputs = model.generate(**inputs, **gen_args)
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summary = post_process_summary(summary)
        chunk_summaries.append(summary)

    # If multiple chunks, re-summarize
    if len(chunk_summaries) > 1:
        combined = " ".join(chunk_summaries)
        meta_prompt = prefix + combined
        inputs = tokenizer(meta_prompt, return_tensors="pt", truncation=True,
                           max_length=1024).to(device)
        # Use a slightly more aggressive penalty for meta-summarization
        meta_gen_args = gen_args.copy()
        meta_gen_args["length_penalty"] = 3.0
        outputs = model.generate(**inputs, **meta_gen_args)
        final_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:
        final_summary = chunk_summaries[0]

    final_summary = post_process_summary(final_summary)
    add_summary(key, model_name, final_summary)

    print(f"\n📝 Legal-T5 (BillSum CA) Summary ({len(final_summary)} chars):")
    print(textwrap.fill(final_summary, width=100))

    del model, tokenizer
    torch.cuda.empty_cache()

except Exception as e:
    print(f"❌ Error: {e}")
    traceback.print_exc()
    add_summary(key, model_name, None, status="failed", error=e)



# =============================================================================
# CELL 11: MODEL - Pegasus-Billsum
# =============================================================================
print("\n" + "="*80)
print("MODEL 8: PEGASUS-BILLSUM")
print("="*80)

model_name = "google/pegasus-billsum"
key = "pegasus_billsum"
prefix = "" # Pegasus models don't typically need a prefix

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    # Pegasus works best with shorter chunks
    chunks = chunk_text(substantive_text, tokenizer, max_len=450, overlap=50)
    print(f"Created {len(chunks)} chunks")

    gen_args = {
        "num_beams": 6,
        "no_repeat_ngram_size": 3,
        "repetition_penalty": 2.0,
        "length_penalty": 2.0,
        "early_stopping": True,
        "max_new_tokens": 200,
    }

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}...")
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True,
                          max_length=512, padding=False).to(device)

        outputs = model.generate(**inputs, **gen_args)
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summary = post_process_summary(summary)
        chunk_summaries.append(summary)

    # If multiple chunks, re-summarize
    if len(chunk_summaries) > 1:
        combined = " ".join(chunk_summaries)
        # Pegasus's input limit is usually smaller, re-chunking meta-summary
        meta_chunks = chunk_text(combined, tokenizer, max_len=450, overlap=50)
        meta_summaries = []
        for meta_chunk in meta_chunks:
            inputs = tokenizer(meta_chunk, return_tensors="pt", truncation=True,
                               max_length=512).to(device)
            outputs = model.generate(**inputs, **gen_args)
            meta_summaries.append(tokenizer.decode(outputs[0], skip_special_tokens=True))
        final_summary = " ".join(meta_summaries)
    else:
        final_summary = chunk_summaries[0]

    final_summary = post_process_summary(final_summary)
    add_summary(key, model_name, final_summary)

    print(f"\n📝 Pegasus-Billsum Summary ({len(final_summary)} chars):")
    print(textwrap.fill(final_summary, width=100))

    del model, tokenizer
    torch.cuda.empty_cache()

except Exception as e:
    print(f"❌ Error: {e}")
    traceback.print_exc()
    add_summary(key, model_name, None, status="failed", error=e)



# =============================================================================
# CELL 12: MODEL - Legal-LED-Base-16384 (Long Context)
# =============================================================================
print("\n" + "="*80)
print("MODEL 9: LEGAL-LED-BASE-16384")
print("="*80)

model_name = "nsi319/legal-led-base-16384"
key = "legal_led_base_16384"
prefix = "" # LED doesn't typically require a prefix

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    # LED can handle longer inputs, use smart_chunk_text
    chunks = smart_chunk_text(substantive_text, tokenizer, max_tokens=2000, overlap_tokens=200)
    print(f"Created {len(chunks)} chunks")

    gen_args = {
        "num_beams": 8,
        "no_repeat_ngram_size": 3,
        "repetition_penalty": 2.5,
        "length_penalty": 2.5,
        "early_stopping": True,
        "max_new_tokens": 600,
    }

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}...")
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True,
                          max_length=4096, padding=False).to(device) # LED input can be up to 16384

        outputs = model.generate(**inputs, **gen_args)
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summary = post_process_summary(summary)
        chunk_summaries.append(summary)

    if len(chunk_summaries) > 1:
        combined = " ".join(chunk_summaries)
        inputs = tokenizer(combined, return_tensors="pt", truncation=True,
                           max_length=2048).to(device) # Max length 2048 for meta-summary input
        outputs = model.generate(**inputs, **gen_args)
        final_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:
        final_summary = chunk_summaries[0]

    final_summary = post_process_summary(final_summary)
    add_summary(key, model_name, final_summary)

    print(f"\n📝 Legal-LED Summary ({len(final_summary)} chars):")
    print(textwrap.fill(final_summary, width=100))

    del model, tokenizer
    torch.cuda.empty_cache()

except Exception as e:
    print(f"❌ Error: {e}")
    traceback.print_exc()
    add_summary(key, model_name, None, status="failed", error=e)




# =============================================================================
# CELL 13: SAVE RESULTS
# =============================================================================
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_json = f"summaries.json"
output_txt = f"legal_summaries_{timestamp}.txt"

# Save JSON
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(summaries_dict, f, indent=2, ensure_ascii=False)

successful = sum(1 for s in summaries_dict['summaries'].values() if s.get('status') == 'success')
failed = sum(1 for s in summaries_dict['summaries'].values() if s.get('status') == 'failed')

print(f"✅ Saved JSON: {output_json}")
print(f"📊 Total: {len(summaries_dict['summaries'])} | Success: {successful} | Failed: {failed}")

# Save readable text
with open(output_txt, 'w', encoding='utf-8') as f:
    f.write("LEGAL TEXT SUMMARIZATION RESULTS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Generated: {datetime.now().isoformat()}\n\n")

    for model_key, data in summaries_dict['summaries'].items():
        f.write(f"\n{'='*80}\n")
        f.write(f"{model_key.upper().replace('_', ' ')}\n")
        f.write(f"{'='*80}\n")
        f.write(f"Model: {data.get('model', 'N/A')}\n")
        f.write(f"Status: {data.get('status')}\n")
        f.write(f"Length: {data.get('summary_length', 0)} chars\n\n")

        if data.get('status') == 'success' and data.get('summary'):
            f.write("SUMMARY:\n")
            f.write(textwrap.fill(data['summary'], width=100) + "\n")
        elif data.get('error'):
            f.write(f"ERROR: {data.get('error')}\n")

print(f"✅ Saved TXT: {output_txt}")

# Display successful summaries
print(f"\n{'='*80}\n📝 SUCCESSFUL SUMMARIES:\n{'='*80}")
for model_key, data in summaries_dict['summaries'].items():
    if data.get('status') == 'success':
        print(f"\n🔹 {model_key.upper()}:")
        print(textwrap.fill(data['summary'], width=100))

print("\n" + "="*80)
print("COMPLETE - Download files from Colab file browser (📁)")
print("="*80)


MODEL 1: T5-BASE


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

...

Created 2 chunks
Processing chunk 1/2...


model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

Processing chunk 2/2...


Input ids are automatically padded from 672 to 1024 to be a multiple of `config.attention_window`: 1024



📝 Legal-LED Summary (1839 chars):
On September 6, 2018, the Honorable Judge of the Patna High Court of Patna entered a final judgment
against an individual who had failed to qualify for selection by the Central Selection Board (CSC).
According to the court's order, Arkshit Kapoor from his remote village went to the Cyber caf at
Pakribarawan-a nearby town on 29.07. after noticing an advertisement issued by the CSC stating that
two or more candidates should correctly mention their date of birth according to their 10th board
certificate. However, as alleged in the complaint, while filling up the form, the application form
got recorded as 08.12 instead of 18.12. In addition, the complaint alleges that after filling out
the application, the appellant was forced to repeat it while submitting the main exam and the
Physical Eligibility Test. The court also found that since the applicant was unaware of his own
mistake he had mechanically signed the printed form. The court granted the relief so

SHORT & LONG SUMMARIZATION

In [ ]:
# =============================================================================
# CELL 0: SETUP - Install and Import
# =============================================================================
print("Installing required packages...")
!pip install -q transformers sentencepiece accelerate datasets torch

Installing required packages...


In [ ]:
import re
import json
import torch
import textwrap
import traceback
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")


# Initialize storage for both types
results_storage = {
    "short_summaries": {},
    "long_summaries": {}
}

def add_summary(category, model_key, model_name, summary_text, status="success", error=None):
    """
    category: 'short_summaries' or 'long_summaries'
    """
    if category not in results_storage:
        results_storage[category] = {}

    results_storage[category][model_key] = {
        "model": model_name,
        "summary": summary_text if status == "success" else None,
        "status": status,
        "error": str(error) if error else None,
        "summary_length": len(summary_text) if summary_text else 0,
        "timestamp": datetime.now().isoformat()
    }

✅ Device: cuda


In [ ]:
# =============================================================================
# CELL 1: DATA LOADING FUNCTIONS (UPDATED FOR GCN JSON STRUCTURE)
# =============================================================================

import os

SHORT_INPUT_FILE = "short_summary_sentences.json"
LONG_INPUT_FILE = "long_summary_sentences.json"

def load_and_join_sentences(filename):
    """
    Loads sentences from the GCN output JSON structure.
    Structure: { "selected": [ {"text": "...", "node_index": 1}, ... ] }
    """
    if not os.path.exists(filename):
        print(f"⚠️ Warning: {filename} not found. Using placeholder text.")
        return "The Plaintiff filed a complaint against the Defendant regarding a breach of contract."

    try:
        with open(filename, 'r') as f:
            data = json.load(f)

        # 1. Access the 'selected' list
        selected_nodes = data.get('selected', [])

        if not selected_nodes:
            print(f"⚠️ Warning: {filename} loaded but 'selected' list is empty.")
            return "The Plaintiff filed a complaint against the Defendant."

        # 2. Sort by 'node_index' to ensure logical reading order
        # (GCNs sometimes extract important sentences out of order)
        selected_nodes.sort(key=lambda x: x.get('node_index', 0))

        # 3. Extract the 'text' field from each node
        sentence_list = [item.get('text', '').strip() for item in selected_nodes]

        # 4. Filter out any empty strings
        sentence_list = [s for s in sentence_list if s]

        # 5. Join them
        joined_text = "\n\n".join(sentence_list)

        print(f"✅ Loaded {filename}: {len(sentence_list)} sentences, {len(joined_text)} chars")
        return joined_text

    except Exception as e:
        print(f"❌ Error loading {filename}: {e}")
        traceback.print_exc()
        return "The Plaintiff filed a complaint against the Defendant regarding a breach of contract."

print("Loading Data...")
raw_text_short = load_and_join_sentences(SHORT_INPUT_FILE)
raw_text_long = load_and_join_sentences(LONG_INPUT_FILE)

Loading Data...
✅ Loaded short_summary_sentences.json: 5 sentences, 1047 chars
✅ Loaded long_summary_sentences.json: 50 sentences, 10502 chars


In [ ]:
# =============================================================================
# CELL 2: HELPER FUNCTIONS (CLEANING & POST-PROCESSING)
# =============================================================================

def clean_legal_boilerplate(text):
    """Remove legal document artifacts"""
    # Case headers
    text = re.sub(r'Case\s+\d+[:\d\w\-\.]+\s+Document\s+\d+.*?Filed.*?\d{4}', '', text, flags=re.I)
    text = re.sub(r'IN THE UNITED STATES DISTRICT.*?DIVISION', '', text, flags=re.I|re.DOTALL)
    text = re.sub(r'Civil Action No\..*?\n', '', text, flags=re.I)
    # Attorney info
    text = re.sub(r'(ATTORNEY|LEAD ATTORNEY|Senior Trial Attorney).*?(?=\n\n|\Z)', '', text, flags=re.I|re.DOTALL)
    text = re.sub(r'(Phone|Telephone|Facsimile|Email|Fax)\s*:?\s*[\(\d\)\s\-\.]+', '', text, flags=re.I)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', '', text)
    # Signatures & Addresses
    text = re.sub(r'Respectfully submitted.*?(?=\n\n|\Z)', '', text, flags=re.I|re.DOTALL)
    text = re.sub(r'/s/.*?\n', '', text)
    text = re.sub(r'\d+\s+\w+\s+(Street|St\.|Avenue|Ave\.|Road|Rd\.|Suite|Ste\.).*?\d{5}', '', text, flags=re.I)
    # Page numbers & Misc
    text = re.sub(r'\bPage\s+\d+\s+of\s+\d+\b', '', text, flags=re.I)
    text = re.sub(r'incorporates?\s+by\s+reference.*?(?=\.|$)', '', text, flags=re.I)
    # Spacing
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def smart_chunk_text(text, tokenizer, max_tokens=800, overlap_tokens=100):
    """Chunk with overlap"""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_tokens:
        return [text]

    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        if end >= len(tokens):
            break
        start = end - overlap_tokens
    return chunks

def post_process_summary(summary):
    """Clean generated summary"""
    # Artifact removal
    summary = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', '', summary)
    summary = re.sub(r'Bar\s+ID\s+\d+', '', summary, flags=re.I)

    # Fragment removal at end
    if summary:
        match = re.search(r'\s+([a-zA-Z]{1,3}\.)?$', summary)
        if match and len(summary.split()) > 10:
            last_full_stop = summary.rfind('.')
            if last_full_stop != -1 and len(summary) - last_full_stop < 50:
                 summary = summary[:last_full_stop + 1]

    # Spacing & Capitalization
    summary = re.sub(r'\s+([.,;:!?])', r'\1', summary)
    summary = re.sub(r'\s{2,}', ' ', summary)

    if summary and summary[0].islower():
        summary = summary[0].upper() + summary[1:]
    if summary and summary[-1] not in '.!?':
        summary += '.'

    return summary.strip()

# Prepare clean texts
cleaned_text_short = clean_legal_boilerplate(raw_text_short)
cleaned_text_long = clean_legal_boilerplate(raw_text_long)

print(f"Short Text Cleaned: {len(cleaned_text_short)} chars")
print(f"Long Text Cleaned: {len(cleaned_text_long)} chars")

Short Text Cleaned: 1043 chars
Long Text Cleaned: 10453 chars


In [ ]:
# =============================================================================
# CELL 3: UNIVERSAL MODEL RUNNER
# =============================================================================

def get_params(target_type, model_type="standard"):
    """
    Returns generation parameters based on short vs long summary.
    model_type: 'standard' (T5/BART) or 'led' (Longformer)
    """
    if target_type == "short":
        # Short Summary Params (approx 150-250 words)
        return {
            "max_new_tokens": 250,
            "min_length": 100,
            "length_penalty": 1.5, # Slightly lower penalty to allow concise endings
            "num_beams": 6,
            "no_repeat_ngram_size": 3,
            "early_stopping": True
        }
    else:
        # Long Summary Params
        return {
            "max_new_tokens": 600, # Previous setting
            "min_length": 200,
            "length_penalty": 2.5, # Encourage length
            "num_beams": 6 if model_type == "standard" else 8,
            "no_repeat_ngram_size": 3,
            "early_stopping": True
        }

def run_pipeline(model_name, key, prefix, use_smart_chunking=False, chunk_size=500):
    print(f"\n🚀 Processing: {model_name} ({key})")

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

        # Iterate over both short and long tasks
        tasks = [
            ("short_summaries", cleaned_text_short, "short"),
            ("long_summaries", cleaned_text_long, "long")
        ]

        for category, source_text, target_type in tasks:
            print(f"   ...generating {target_type} summary")

            # 1. Determine Parameters
            gen_args = get_params(target_type, model_type="led" if "led" in key or "long" in key else "standard")

            # 2. Chunking Strategy
            # Adjust chunk size specifically for LED/LongT5 vs Standard
            current_chunk_size = chunk_size
            if target_type == "long" and ("led" in key or "long" in key):
                current_chunk_size = 2000 # Larger chunks for long context models

            if use_smart_chunking:
                chunks = smart_chunk_text(source_text, tokenizer, max_tokens=current_chunk_size, overlap_tokens=100)
            else:
                # Basic chunking fallback logic inside here for consistency
                tokens = tokenizer.encode(source_text, add_special_tokens=False)
                chunks = []
                for i in range(0, len(tokens), current_chunk_size):
                    chunks.append(tokenizer.decode(tokens[i:i + current_chunk_size]))

            # 3. Generation Loop
            chunk_summaries = []
            for chunk in chunks:
                prompt = prefix + chunk if prefix else chunk

                # Input length handling
                max_input_len = 4096 if ("led" in key or "long" in key) else 512
                if key == "bart_large_cnn": max_input_len = 1024

                inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_len).to(device)

                # Generate
                outputs = model.generate(**inputs, **gen_args)
                text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                chunk_summaries.append(post_process_summary(text))

            # 4. Meta-Summarization (if multiple chunks)
            if len(chunk_summaries) > 1:
                combined = " ".join(chunk_summaries)
                # For short summaries, we want the final combine to be very strict on length
                if target_type == "short":
                    # Be even stricter for the final pass
                    final_gen_args = gen_args.copy()
                    final_gen_args["max_new_tokens"] = 250
                    final_gen_args["min_length"] = 80
                else:
                    final_gen_args = gen_args

                meta_prompt = prefix + combined if prefix else combined

                # Check Input Limit for Meta Summary
                max_meta_len = 2048 if ("led" in key or "long" in key) else 1024 # Standard models might struggle here, truncate if needed
                if key in ["t5-base", "flan_t5_base", "pegasus_xsum"]: max_meta_len = 512

                inputs = tokenizer(meta_prompt, return_tensors="pt", truncation=True, max_length=max_meta_len).to(device)
                outputs = model.generate(**inputs, **final_gen_args)
                final_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
            else:
                final_summary = chunk_summaries[0]

            final_summary = post_process_summary(final_summary)
            add_summary(category, key, model_name, final_summary)

            # Print preview
            print(f"      -> Length: {len(final_summary)} chars")

        del model, tokenizer
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ Error in {model_name}: {e}")
        traceback.print_exc()
        add_summary("short_summaries", key, model_name, None, "failed", error=e)
        add_summary("long_summaries", key, model_name, None, "failed", error=e)

In [ ]:
# =============================================================================
# CELL 4: EXECUTE ALL MODELS
# =============================================================================

print("\n" + "="*80)
print("STARTING BATCH PROCESSING")
print("="*80)

# 1. T5-BASE
run_pipeline("t5-base", "t5-base", "summarize: Provide a legal summary. ", use_smart_chunking=False, chunk_size=500)

# 2. BART-LARGE-CNN
run_pipeline("facebook/bart-large-cnn", "bart_large_cnn", "", use_smart_chunking=False, chunk_size=600)

# 3. PEGASUS-XSUM
run_pipeline("google/pegasus-xsum", "pegasus_xsum", "", use_smart_chunking=False, chunk_size=450)

# 4. LED-BASE-16384 (Long Context)
# Uses smart chunking inside the function based on type
run_pipeline("allenai/led-base-16384", "led_base_16384", "", use_smart_chunking=True, chunk_size=2000)

# 5. LEGAL-PEGASUS (Best Performer)
run_pipeline("nsi319/legal-pegasus", "legal_pegasus", "Generate a detailed summary of the legal case. ", use_smart_chunking=True, chunk_size=450)

# 6. LONG-T5-TGLOBAL-BASE
run_pipeline("google/long-t5-tglobal-base", "long_t5_tglobal_base", "summarize: Provide a detailed legal summary. ", use_smart_chunking=True, chunk_size=1500)

# 7. FLAN-T5-BASE
run_pipeline("google/flan-t5-base", "flan_t5_base", "Summarize this legal text: ", use_smart_chunking=False, chunk_size=500)

# 8. LEGAL-T5 (BILLSUM)
run_pipeline("stevhliu/t5-small-finetuned-billsum-ca_test", "legal_t5_billsum", "summarize: Provide a legal summary. ", use_smart_chunking=False, chunk_size=500)

# 9. PEGASUS-BILLSUM
run_pipeline("google/pegasus-billsum", "pegasus_billsum", "", use_smart_chunking=False, chunk_size=450)

# 10. LEGAL-LED-BASE
run_pipeline("nsi319/legal-led-base-16384", "legal_led_base_16384", "", use_smart_chunking=True, chunk_size=2000)


STARTING BATCH PROCESSING

🚀 Processing: t5-base (t5-base)


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

   ...generating short summary
      -> Length: 388 chars
   ...generating long summary
      -> Length: 571 chars

🚀 Processing: facebook/bart-large-cnn (bart_large_cnn)


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

   ...generating short summary
      -> Length: 462 chars
   ...generating long summary
      -> Length: 1019 chars

🚀 Processing: google/pegasus-xsum (pegasus_xsum)


...

Loading weights:   0%|          | 0/299 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie led.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


   ...generating short summary
      -> Length: 1207 chars
   ...generating long summary


Input ids are automatically padded from 704 to 1024 to be a multiple of `config.attention_window`: 1024


      -> Length: 2438 chars


In [ ]:
# =============================================================================
# CELL 5: SAVE RESULTS
# =============================================================================
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_json = f"dual_summaries_{timestamp}.json"
output_txt = f"dual_summaries_report_{timestamp}.txt"

# Save JSON
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(results_storage, f, indent=2, ensure_ascii=False)

print(f"✅ Saved JSON: {output_json}")

# Save readable text report
with open(output_txt, 'w', encoding='utf-8') as f:
    f.write("LEGAL TEXT SUMMARIZATION REPORT (SHORT & LONG)\n")
    f.write("="*80 + "\n")
    f.write(f"Generated: {datetime.now().isoformat()}\n\n")

    # Section 1: Short Summaries
    f.write(f"\n{'#'*30} SHORT SUMMARIES (Target 150-250) {'#'*30}\n")
    for key, data in results_storage['short_summaries'].items():
        f.write(f"\n{'='*80}\nMODEL: {key.upper()}\n{'='*80}\n")
        if data['status'] == 'success':
            f.write(textwrap.fill(data['summary'], width=100))
            f.write(f"\n\n[Length: {data['summary_length']}]\n")
        else:
            f.write(f"FAILED: {data['error']}\n")

    # Section 2: Long Summaries
    f.write(f"\n\n{'#'*30} LONG SUMMARIES (Target 600+) {'#'*30}\n")
    for key, data in results_storage['long_summaries'].items():
        f.write(f"\n{'='*80}\nMODEL: {key.upper()}\n{'='*80}\n")
        if data['status'] == 'success':
            f.write(textwrap.fill(data['summary'], width=100))
            f.write(f"\n\n[Length: {data['summary_length']}]\n")
        else:
            f.write(f"FAILED: {data['error']}\n")

print(f"✅ Saved TXT: {output_txt}")
print("\nComplete. Check the files tab.")


SAVING RESULTS
✅ Saved JSON: dual_summaries_20260212_182451.json
✅ Saved TXT: dual_summaries_report_20260212_182451.txt

Complete. Check the files tab.
